In [1]:
import cv2
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
import easyocr

# Download YOLO license plate model from Hugging Face
model_path = hf_hub_download("koushik-ai/yolov8-license-plate-detection", "best.pt")
model = YOLO(model_path)

# EasyOCR reader (English only for speed)
reader = easyocr.Reader(['en'])

# Open webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detect plates
    results = model(frame, verbose=False)
    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        conf = float(box.conf[0])
        if conf < 0.5:
            continue

        # Crop license plate
        plate_img = frame[y1:y2, x1:x2]

        # OCR
        ocr_result = reader.readtext(plate_img)
        if ocr_result:
            plate_number = ocr_result[0][-2]  # Extract text
        else:
            plate_number = ""

        # Draw detection + text
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, plate_number, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("License Plate Detection", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


RepositoryNotFoundError: 401 Client Error. (Request ID: Root=1-6899a675-369b198706499f1b28fb241d;32bd671a-9be6-45fc-be28-d46473cfd1c8)

Repository Not Found for url: https://huggingface.co/koushik-ai/yolov8-license-plate-detection/resolve/main/best.pt.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated.
Invalid username or password.